# ECON2041 Week 11 tutorial: testing several coefficients at once

### What you'll be able to do by the end

- Read the standard error, the t statistic, and the 95% confidence interval of each coefficient in a regression table
- State a null hypothesis about two or more coefficients at once
- Compute an F statistic by hand from two sums of squared residuals, and check it with `f_test`
- Explain why a joint test can reject the null hypothesis when the t-test on one of its coefficients does not

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running
- ⭐ Optional: extra exercise if you have time

## Setup

Our usual setup block, plus the two extra lines from the week 9 tutorial, `glob` and `os`, which the upload cell below uses to find your .csv file.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression
import glob                      # finds files by their name
import os                        # file tools

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## Upload the recordings dataset again

We'll upload the dataset from the week 9 tutorial again, in the same four steps:

1. Download `econ2041-recordings-quiz.csv` from the week 9 tutorial page on iLearn, unless you still have it from the week 9 tutorial
2. Click the folder icon in Colab's left sidebar to open the Files panel
3. Drag the .csv file into that panel, or click the upload icon at the top of the panel and find the file wherever your browser saved it
4. Run the cell below

In [ ]:
# 🟢 Read and run: load econ2041-recordings-quiz.csv into a dataframe called recordings_quiz
# look for the .csv file in Colab's Files panel, newest first
found = set(
    glob.glob("econ2041-recordings-quiz*.csv")
    + glob.glob("/content/econ2041-recordings-quiz*.csv")
    + glob.glob("/econ2041-recordings-quiz*.csv")
)

if found:
    csv = max(found, key=os.path.getmtime)
    recordings_quiz = pd.read_csv(csv)
    print("Loaded", csv)
    display(recordings_quiz.head())
else:
    print("No econ2041-recordings-quiz .csv file yet. Please follow the instructions above to upload the dataset.")

### What's in this dataset

It's the dataset from the week 9 tutorial: one row for each of the 209 ECON2041 students who sat the week 7 quiz and appear in the Echo360 viewing records.

- `recordings`: how many of the six live lecture recordings from weeks 1 to 6 the student watched at least part of on Echo360
- `hours`: the total time that the student spent watching those six recordings on Echo360, in hours
- `quiz`: the student's week 7 quiz mark out of 20

As in the week 9 tutorial, the data are anonymous: we removed all names and student IDs, put the rows in random order, and added a small random number to every quiz mark.

*Source:* Echo360 viewing records for the ECON2041 live lectures, from iLearn Insights, and the ECON2041 week 7 quiz marks.

## Two measures of watching in one regression

### Question 1: is each measure associated with quiz marks, holding the other fixed?

In the week 9 tutorial, we regressed `quiz` on `hours`.
The cell below fits that regression again.

In [ ]:
# 🟢 Read and run: regress quiz on hours, as in the week 9 tutorial, and show the regression table rounded to 3 decimals
hours_fit = ols("quiz ~ hours", data=recordings_quiz).fit()

hours_fit.summary2().tables[1].round(3)

In [ ]:
# ✏️ You write: run a regression of quiz on recordings and hours together, store the result as both_fit,
# and show the regression table rounded to 3 decimals
# Hint: two explanatory variables to the right of ~, separated by +; the table comes from .summary2().tables[1],
#       as in the cell above

#### Read the whole regression table

In the week 9 tutorial, we used the `Coef.` and `P>|t|` columns.
The other columns are from the week 9 Essential Concepts recordings:

- `Std.Err.`: the standard error, our estimate, from one sample, of how much the coefficient would vary across samples
- `t`: the t statistic, the coefficient divided by its standard error, so how far the estimate is from 0, measured in standard errors
- `[0.025` and `0.975]`: the lower and upper bounds of the 95% confidence interval

A 95% confidence interval excludes 0 exactly when the p-value is below 0.05.

✏️ Complete the sentences, using the table for `both_fit`:

- Holding hours fixed, each additional recording watched is associated with a quiz mark that is ________ marks higher, on average. The p-value is ________, and the 95% confidence interval has a lower bound of ________ and an upper bound of ________
- Holding recordings fixed, each additional hour watched is associated with a quiz mark that is ________ marks higher, on average. The p-value is ________, and the 95% confidence interval has a lower bound of ________ and an upper bound of ________

🤔 At the unit's threshold of 0.1, can we reject the null hypothesis that the coefficient on `recordings` is 0? And the null hypothesis that the coefficient on `hours` is 0?
Does each 95% confidence interval contain 0?

## Test both measures at once

### Question 2: is watching, by either measure, associated with quiz marks?

Our regression model with both measures is

$$\text{quiz}_i = \beta_0 + \beta_1 \, \text{recordings}_i + \beta_2 \, \text{hours}_i + u_i$$

The joint null hypothesis is that both slope coefficients are 0:

$$H_0: \beta_1 = \beta_2 = 0$$

The alternative is that at least one of the two coefficients is not 0.
As in the week 10 live lecture, the F test compares two models:

- Unrestricted: the model above, which we estimated as `both_fit`
- Restricted: the model with the null hypothesis imposed, so with both slopes set to 0: $\text{quiz}_i = \beta_0 + u_i$

$$F = \frac{(SSR_r - SSR_{ur})/Q}{SSR_{ur}/(n-k-1)}$$

$SSR_r$ and $SSR_{ur}$ are the sums of squared residuals of the restricted and unrestricted models, $Q$ is the number of restrictions in the null hypothesis, $n$ is the number of observations, and $k$ is the number of slope coefficients in the unrestricted model.

In [ ]:
# ✏️ You write: run the restricted regression of quiz on an intercept only, store the result as restricted_fit,
# and show the coefficients rounded to 2 decimals
# Hint: with no explanatory variables, the formula is "quiz ~ 1", where the 1 asks for an intercept only;
#       the coefficients are in restricted_fit.params

In [ ]:
# 🟢 Read and run: the sums of squared residuals of the restricted and unrestricted models, rounded to 1 decimal
restricted_fit.ssr.round(1), both_fit.ssr.round(1)

In [ ]:
# ✏️ You write: compute F by hand and round it to 2 decimals, with Q = 2 and n - k - 1 = 209 - 2 - 1 = 206
# Hint: the restricted SSR is restricted_fit.ssr and the unrestricted SSR is both_fit.ssr;
#       put the top and the bottom of the formula in their own parentheses, then add .round(2) to the whole thing

In [ ]:
# ✏️ You write: run the joint test of the null hypothesis that the coefficients on recordings and hours are both 0, with f_test on both_fit
# Hint: f_test takes both restrictions inside one pair of quotation marks, each written as the variable name = 0,
#       separated by a comma

🤔 Does `f_test` give the same F as your calculation?
Is the p-value below 0.1, and what do we conclude about $H_0: \beta_1 = \beta_2 = 0$?

💬 With your neighbors: in `both_fit`, the coefficient on `recordings` was not distinguishable from 0, and the p-value on `hours` was only just below 0.1.
Yet the joint test rejects the null hypothesis with a p-value of 0.005.
How can both be true?
Hint: the correlation between `recordings` and `hours` is 0.73.

## Back to the Sydney Airbnb listings

### Question 3: does location matter, holding size and room type fixed?

In the week 8 tutorial, we kept the listings in three areas, Parramatta, Sydney, and Waverley, and added the area to the regression as a block of dummies.
The four cells below repeat those steps.

*Source:* [Inside Airbnb](https://insideairbnb.com/), Sydney, snapshot of 16 June 2026, CC BY 4.0.

In [ ]:
# 🟢 Read and run: load sydney-airbnb-teaching.csv into a dataframe called airbnb and look at the first few rows
airbnb = pd.read_csv(f"{DATA}/sydney-airbnb-teaching.csv")

airbnb.head()

In [ ]:
# 🟢 Read and run: build the entire dummy from the week 7 tutorial
#                  this takes a value of 1 for an entire home or apartment
#                  and a value of 0 for any kind of "room"
airbnb["entire"] = (airbnb["room_type"] == "Entire home/apt").astype(int)

airbnb["entire"].mean().round(2)

In [ ]:
# 🟢 Read and run: keep the listings in three areas in a smaller dataframe called three, and count them
three = airbnb[airbnb["neighbourhood"].isin(["Parramatta", "Sydney", "Waverley"])]

three["neighbourhood"].value_counts()

In [ ]:
# 🟢 Read and run: regress price on accommodates, bedrooms, entire, and the area dummies, as in the week 8 tutorial,
#                  and show the regression table rounded to 3 decimals
areas = ols("price ~ accommodates + bedrooms + entire + C(neighbourhood)", data=three).fit()

areas.summary2().tables[1].round(3)

🤔 Write the null hypothesis that location doesn't matter, holding guests, bedrooms, and room type fixed.
Which coefficients does it set to 0, and what does it say about prices in the three areas?

In [ ]:
# ✏️ You write: run the joint test of the null hypothesis that both area coefficients are 0, with f_test on areas
# Hint: in f_test, write each coefficient's name exactly as its row label in the table above,
#       including C(neighbourhood) and the square brackets

### ⭐ Question 4 (optional): all 38 areas at once

In the week 8 tutorial's optional question 5, we fit the same model on the full `airbnb` file, with 38 areas, 37 dummy variables, and Ashfield as the reference category.
Writing 37 restrictions inside `f_test` would take a long line, so we'll compute F by hand instead.

In [ ]:
# 🟢 Read and run: fit the model with all 38 areas, store the result as areas_all,
#                  and count the area coefficients that have a p-value below 0.1 on their own
areas_all = ols("price ~ accommodates + bedrooms + entire + C(neighbourhood)", data=airbnb).fit()

(areas_all.pvalues.filter(like="neighbourhood") < 0.1).sum()

In [ ]:
# 🟢 Read and run: fit the model without the area dummies, store the result as no_areas,
#                  and show the sums of squared residuals of no_areas and areas_all
no_areas = ols("price ~ accommodates + bedrooms + entire", data=airbnb).fit()

no_areas.ssr.round(), areas_all.ssr.round()

In [ ]:
# ✏️ You write (optional): compute F by hand for the null hypothesis that all 37 area coefficients are 0, rounded to 2 decimals
# Hint: the same formula as in question 2, with no_areas.ssr as the restricted SSR and areas_all.ssr as the unrestricted SSR;
#       Q = 37, and the unrestricted model has 3 + 37 = 40 slope coefficients, so n - k - 1 = 12,419 - 40 - 1 = 12,378

🤔 19 of the 37 area coefficients have a p-value below 0.1 on their own, and you have just computed F.
Which of the two answers the question "does location matter, holding size and room type fixed?"
Why not count the area coefficients with a p-value below 0.1 instead?

### ⭐ Question 5 (optional): one restriction

In the week 8 tutorial's question 3, we let the guests slope differ between rooms and entire homes with an interaction term:

$$\text{price}_i = \beta_0 + \beta_1 \, \text{accommodates}_i + \beta_2 \, \text{entire}_i + \beta_3 \, (\text{accommodates}_i \times \text{entire}_i) + u_i$$

The null hypothesis that the slopes are the same is $H_0: \beta_3 = 0$, a single restriction, so $Q = 1$.

In [ ]:
# 🟢 Read and run: fit the interaction model from the week 8 tutorial, store the result as inter,
#                  and show the regression table rounded to 3 decimals
inter = ols("price ~ accommodates * entire", data=airbnb).fit()

inter.summary2().tables[1].round(3)

In [ ]:
# ✏️ You write (optional): run the test of the null hypothesis that the interaction coefficient is 0, with f_test on inter
# Hint: one restriction inside the quotation marks, written as the interaction term's row label from the table above = 0

🤔 Compare F with the t statistic on `accommodates:entire` in the table: what is $t^2$?
When the null hypothesis has only one restriction, do we need an F test?